# Milestone 1 – CIE
## Data Transformation, Null Value Handling, Insight Generation, and Visualization

**Course/Section:** 4SF25CI111_3B  
**Dataset:** Netflix Movies and TV Shows

### Objective
Analyze a real-world CSV dataset using Python and Pandas. Understand the data, transform columns, handle numerical missing values using mean imputation, generate insights, and visualize the results.

### Dataset Information
The Netflix Movies and TV Shows dataset contains information about titles available on Netflix, including content type, title, director, cast, country, date added, release year, rating, duration, genre, and description.

- **Dataset file:** `netflix_titles.csv`
- **Dataset size:** Approximately 8,800 records and 12 columns (the exact count is displayed after loading).
- **Original dataset:** [Netflix Movies and TV Shows – Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)
- **CSV mirror used by this notebook:** [Download CSV](https://raw.githubusercontent.com/minhaj-313/Netflix-Movies-and-TV-Shows-Dashboard/main/netflix_titles.csv)

The notebook reads the CSV directly from the public link, so a separate manual download is not required. An internet connection is required for the first data-loading step.

### Libraries Used
- Python
- Pandas
- NumPy
- Matplotlib


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all columns when viewing DataFrames
pd.set_option("display.max_columns", None)


## 2. Dataset Selection and Reading
Read the CSV file into a Pandas DataFrame.

In [ ]:
# Read the Netflix dataset from the public CSV source
csv_url = "https://raw.githubusercontent.com/minhaj-313/Netflix-Movies-and-TV-Shows-Dashboard/main/netflix_titles.csv"

df = pd.read_csv(csv_url)

print("Dataset loaded successfully!")
print("Number of rows and columns:", df.shape)
display(df.head())


## 3. Data Understanding
Check the dataset structure, column names, data types, and basic statistical information.

In [ ]:
print("Dataset information:")
df.info()

print("\nColumn names:")
print(df.columns.tolist())

print("\nStatistical summary:")
display(df.describe(include="all").T)

print("\nFirst five records:")
display(df.head())

print("\nLast five records:")
display(df.tail())


## 4. Identify Null Values
Count the missing values in every column and calculate their percentages.

In [ ]:
null_counts = df.isnull().sum()
null_percent = (df.isnull().mean() * 100).round(2)

null_summary = pd.DataFrame({
    "Null Count": null_counts,
    "Null Percentage": null_percent
}).sort_values("Null Count", ascending=False)

display(null_summary)


## 5. Data Transformation
Perform useful transformations:
- Remove columns that are not required for this analysis (`show_id` and `description`).
- Convert `date_added` into a datetime column.
- Extract the numeric part of `duration` into a new column.
- Create `duration_unit` to identify minutes or seasons.
- Extract the year and month when a title was added.

In [ ]:
# Work on a copy so the original DataFrame remains unchanged
data = df.copy()

# Drop columns not needed for the selected analysis
data.drop(columns=["show_id", "description"], inplace=True, errors="ignore")

# Convert date_added to datetime; invalid or missing dates become NaT
data["date_added"] = pd.to_datetime(data["date_added"].str.strip(), errors="coerce")

# Extract numeric duration and its unit
data["duration_value"] = pd.to_numeric(
    data["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

data["duration_unit"] = np.where(
    data["type"].eq("Movie"), "Minutes", "Seasons"
)

# Extract year and month from the date_added column
data["year_added"] = data["date_added"].dt.year
data["month_added"] = data["date_added"].dt.month

print("Data after transformation:")
display(data.head())
print("\nUpdated data types:")
print(data.dtypes)


## 6. Handling Null Values Using Mean Imputation
Mean imputation is suitable for numerical columns. The `duration_value` column contains movie durations in minutes and TV show durations in seasons, so calculate the mean separately for Movies and TV Shows. This avoids mixing different units. Also fill missing `year_added` values with its numerical mean (rounded to the nearest year). Categorical columns are filled with `Unknown`, not a numerical mean.

In [ ]:
print("Missing values before imputation:")
display(data.isnull().sum())

# Mean imputation within each content type:
# Movie duration is measured in minutes; TV Show duration is measured in seasons.
data["duration_value"] = data.groupby("type")["duration_value"].transform(
    lambda values: values.fillna(values.mean())
)

# Fill any remaining duration gaps with the overall numeric mean as a fallback
data["duration_value"] = data["duration_value"].fillna(
    data["duration_value"].mean()
)

# Mean imputation for the numerical year_added column
mean_year = data["year_added"].mean()
if pd.notna(mean_year):
    data["year_added"] = data["year_added"].fillna(round(mean_year))

# Fill missing categorical values with a label
categorical_columns = data.select_dtypes(include=["object"]).columns
data[categorical_columns] = data[categorical_columns].fillna("Unknown")

print("\nMissing values after imputation:")
display(data.isnull().sum())


## 7. Verify the Cleaned Dataset
Check the cleaned data, duplicate records, and updated structure.

In [ ]:
print("Cleaned dataset shape:", data.shape)

print("\nDuplicate rows:", data.duplicated().sum())

print("\nCleaned dataset preview:")
display(data.head())

print("\nCleaned dataset information:")
data.info()


## 8. Insight Generation
Generate useful summaries about Movies and TV Shows, release years, countries, and genres.

In [ ]:
# Insight 1: Number of Movies and TV Shows
content_counts = data["type"].value_counts()
print("Content type counts:")
display(content_counts)

# Insight 2: Top 10 countries by number of titles
# A row can contain more than one country, so split the country values.
country_counts = (
    data.loc[data["country"] != "Unknown", "country"]
    .str.split(", ")
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
)
print("\nTop 10 countries:")
display(country_counts)

# Insight 3: Titles by release year
release_year_counts = data["release_year"].value_counts().sort_index()
print("\nMost recent release years by number of titles:")
display(release_year_counts.tail(10))

# Insight 4: Most common genres
genre_counts = (
    data["listed_in"]
    .loc[data["listed_in"] != "Unknown"]
    .str.split(", ")
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
)
print("\nTop 10 genres:")
display(genre_counts)

# Insight 5: Average duration value by content type
average_duration = data.groupby("type")["duration_value"].mean().round(2)
print("\nAverage duration value by content type:")
display(average_duration)


## 9. Data Visualization
Use Matplotlib to visualize the main findings.

### 9.1 Movies vs TV Shows

In [ ]:
plt.figure(figsize=(7, 5))
content_counts.plot(kind="bar")
plt.title("Number of Movies and TV Shows")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 9.2 Top 10 Countries

In [ ]:
plt.figure(figsize=(10, 5))
country_counts.sort_values().plot(kind="barh")
plt.title("Top 10 Countries by Number of Netflix Titles")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


### 9.3 Titles by Release Year

In [ ]:
yearly_titles = data["release_year"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
yearly_titles.plot(kind="line")
plt.title("Netflix Titles by Release Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.grid(True)
plt.tight_layout()
plt.show()


### 9.4 Top 10 Genres

In [ ]:
plt.figure(figsize=(10, 5))
genre_counts.sort_values().plot(kind="barh")
plt.title("Top 10 Genres on Netflix")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()


### 9.5 Movies and TV Shows Added by Year

In [ ]:
added_by_year = (
    data.dropna(subset=["date_added"])
    .groupby(["year_added", "type"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

plt.figure(figsize=(12, 5))
for content_type in added_by_year.columns:
    plt.plot(added_by_year.index, added_by_year[content_type], marker="o", label=content_type)

plt.title("Netflix Movies and TV Shows Added by Year")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 10. Final Observations and Conclusion
The following code prints observations based on the actual dataset loaded. Review the output and use it to describe the findings in your own words.

In [ ]:
most_common_type = content_counts.idxmax()
most_common_country = country_counts.idxmax() if not country_counts.empty else "Not available"
most_common_genre = genre_counts.idxmax() if not genre_counts.empty else "Not available"
latest_year = int(data["release_year"].max())

print("FINAL OBSERVATIONS")
print("------------------")
print(f"1. The dataset contains {len(data)} records after loading.")
print(f"2. The most common content type is: {most_common_type}.")
print(f"3. The country with the most listed titles is: {most_common_country}.")
print(f"4. The most common genre is: {most_common_genre}.")
print(f"5. The latest release year in the dataset is: {latest_year}.")
print("\nConclusion:")
print("The analysis explored Netflix content using Pandas. Data transformation and")
print("mean imputation were performed on applicable numerical columns. Grouping,")
print("summaries, and visualizations helped identify patterns in content types,")
print("countries, genres, and release years.")


## 11. Result

The Netflix CSV dataset was successfully loaded and explored using Pandas. Data transformations were performed, numerical null values were handled using mean imputation, meaningful insights were generated, and the results were visualized using Matplotlib.

## References
- [Kaggle – Netflix Movies and TV Shows Dataset](https://www.kaggle.com/datasets/shivamb/netflix-shows)
- [CSV dataset mirror](https://raw.githubusercontent.com/minhaj-313/Netflix-Movies-and-TV-Shows-Dashboard/main/netflix_titles.csv)
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Matplotlib Documentation](https://matplotlib.org/stable/)
